In [1]:
%%capture --no-stderr
%pip install --upgrade --quiet  langchain langchain-community langchainhub langchain-chroma beautifulsoup4


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\msamet\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [14]:
%%capture
! pip install --upgrade langchain-groq

In [3]:
%%capture
! pip install langchain_google_genai gradio

In [4]:
%%capture
! pip install -U chroma

In [ ]:
import getpass
import os
from dotenv import load_dotenv
load_dotenv() 
if "GROQ_API_KEY" not in os.environ:
   GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if "GOOGLE_API_KEY" not in os.environ:
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


In [12]:
!pip install langchain-groq

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\msamet\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
)

C:\Users\msamet\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough


def create_retrieval_chain(retriever, combine_documents_chain):
    """Helper function to create a retrieval chain"""
    return (
        RunnableParallel(
            {"context": retriever, "input": RunnablePassthrough()}
        )
        | combine_documents_chain
    )

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Using HuggingFace embeddings as an alternative
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

C:\Users\msamet\AppData\Local\Temp\ipykernel_6800\1055727712.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


In [20]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data_esb.txt", encoding="utf-8")
docs = loader.load()
# Augmenter la taille des chunks pour conserver plus de contexte
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
splits = text_splitter.split_documents(docs)
print(f"Nombre de chunks: {len(splits)}")

Nombre de chunks: 280


In [24]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
# Récupérer plus de documents pour avoir plus de contexte
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}  # Récupérer 8 chunks au lieu de 4
)

In [ ]:
# Installation de BM25 pour la recherche hybride
%%capture
! pip install rank-bm25

In [ ]:
# Créer un retriever hybride (BM25 + Semantic Search)
from rank_bm25 import BM25Okapi
from langchain.retrievers import EnsembleRetriever
from langchain.retrievers.bm25 import BM25Retriever

# Créer le retriever BM25 pour la recherche par mots-clés
bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 5

# Créer le retriever sémantique (vectorielle)
semantic_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# Combiner les deux retrievers
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.5, 0.5]  # Équilibre entre BM25 et recherche sémantique
)

print("✅ Retriever hybride (BM25 + Semantic) créé avec succès!")

In [25]:
# 2. Incorporate the retriever into a question-answering chain.
from langchain_core.output_parsers import StrOutputParser

system_prompt = (
    """Tu es un assistant académique utile et multilingue travaillant pour l'ESB (Esprit School of Business).
Tu DOIS répondre en te basant UNIQUEMENT sur le contexte fourni.
Si une information n'est pas dans le contexte, dis clairement que tu n'as pas cette information.

Catégorise toujours les programmes en:
- 'Licenses' (Bachelor's degrees / Licences)
- 'Masters' (Master's degrees / Masters)

Sois précis, détaillé et utilise UNIQUEMENT les informations exactes du contexte fourni.

Context: {context}

Question: {question}

Answer:"""
)

prompt = ChatPromptTemplate.from_template(system_prompt)

# Create a simple QA chain
question_answer_chain = prompt | llm | StrOutputParser()

# Create the RAG chain
def create_simple_rag_chain(retriever, qa_chain):
    """Create a simple RAG chain"""
    def rag_chain(query):
        docs = retriever.invoke(query)
        context = "\n".join([doc.page_content for doc in docs])
        result = qa_chain.invoke({"context": context, "question": query})
        return result
    return rag_chain

rag_chain = create_simple_rag_chain(retriever, question_answer_chain)

In [18]:
response = rag_chain("What are the study programs provided by ESB? Provide a full explanation")
print("Response:")
print(response)

Response:
I'm ready to help. What's your question about the academic tracks at ESprit School of Business (ESB)?


In [27]:
# Test le retriever avec la requête EXACTE de l'utilisateur
query = "quels sont les masters disponible dans esb"
docs = retriever.invoke(query)
print(f"Nombre de documents récupérés: {len(docs)}")
print("\n=== DOCUMENTS RÉCUPÉRÉS ===\n")
for i, doc in enumerate(docs, 1):
    print(f"Document {i}:")
    print(doc.page_content[:500])  # Affiche les premiers 500 caractères
    print("---\n")

# Maintenant testons le RAG chain complet
print("\n\n=== TEST DU RAG CHAIN COMPLET ===\n")
response = rag_chain(query)
print("Réponse du RAG chain:")
print(response)

Nombre de documents récupérés: 8

=== DOCUMENTS RÉCUPÉRÉS ===

Document 1:
de fin d’études L’accès au Master : ']
---

Document 2:
de manière efficace.', 'Concilier et gérer les confits.', 'Animer et coordonner l’activité d’une équipe.']
---

Document 3:
'Appréhender les spécificités des technologies liées au web..']
---

Document 4:
CONDITIONS D'EXERCICE DE L'ACTIVITÉ; competence : COMPÉTENCES DE BASE
SAVOIR FAIRE
Réceptionner les documents à traiter, les vérifier et s'informer des consignes de délai, de nombre, ...
Saisir ou numériser les documents (textes manuscrits, chèques, ...)
Prendre des notes sous la dictée ou consigner intégralement les propos tenus en réunion lors de conférences, séances
---

Document 5:
Peut commander des matières premières.
Dirige un service.
ACCÈS À L'EMPLOI MÉTIER
---

Document 6:
Peut superviser un projet maîtrise d'ouvrage.
ACCÈS À L'EMPLOI MÉTIER
---

Document 7:
Peut superviser un projet maîtrise d'ouvrage.
ACCÈS À L'EMPLOI MÉTIER
---

Document 8:
Pe

In [28]:
# Test retriever hybride pour toutes les questions
test_queries = [
    "quels sont les licences disponibles",
    "quels sont les matieres du master gamma",
    "quels sont les programmes d'études"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}")
    
    try:
        docs = hybrid_retriever.invoke(query)
        print(f"Documents trouvés: {len(docs)}\n")
        for i, doc in enumerate(docs[:3], 1):
            print(f"Doc {i}: {doc.page_content[:300]}")
            print("---")
    except Exception as e:
        print(f"ERROR: {e}")


QUERY: quels sont les licences disponibles
ERROR: name 'hybrid_retriever' is not defined

QUERY: quels sont les matieres du master gamma
ERROR: name 'hybrid_retriever' is not defined

QUERY: quels sont les programmes d'études
ERROR: name 'hybrid_retriever' is not defined


In [15]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough

# Create a prompt that reformulates the question based on chat history
contextualize_q_system_prompt = (
    """
Given a chat history and the latest user question which might reference context in the chat history, 
formulate a standalone question which can be understood without the chat history. 
Do NOT answer the question, just reformulate it if needed and otherwise return it as is.
You are a helpful academic assistant and advisor. You understand and answer questions in French, English, Arabic, and Spanish.
"""
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# Create a chain that reformulates the question
history_aware_chain = contextualize_q_prompt | llm

# This retriever will reformulate the question based on history before retrieving
def contextualize_retriever(input_dict):
    """Contextualize the question based on chat history"""
    question = input_dict.get("input", "")
    chat_history = input_dict.get("chat_history", [])
    
    if chat_history:
        contextualized = history_aware_chain.invoke({
            "input": question,
            "chat_history": chat_history
        })
        # Extract the text from the response
        if hasattr(contextualized, 'content'):
            question = contextualized.content
        else:
            question = str(contextualized)
    
    return retriever.invoke(question)

history_aware_retriever = contextualize_retriever

In [ ]:
## How to Run Streamlit App

To run this chatbot application with Streamlit, create a file named `app.py` with the following content and run:

```bash
streamlit run app.py
```

**First, install Streamlit if you haven't already:**
```bash
pip install streamlit
```

The Streamlit app provides:
- 🎓 Clean web interface
- 🌐 Multilingual support (English, French, Arabic)
- 💬 Chat history management
- 🔄 Language switching capability
- 🎯 Real-time RAG responses
- ⚙️ Settings sidebar with options

In [ ]:
import streamlit as st
from langchain_core.messages import AIMessage, HumanMessage

# Set page config
st.set_page_config(
    page_title="ESB Academic Assistant",
    page_icon="🎓",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Initialize session state
if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "assistant", "content": "Hello! I am your multilingual assistant. I can assist you in English, French, and Arabic. Please choose your preferred language: English, Français, or عربي"}
    ]

if "language_preference" not in st.session_state:
    st.session_state.language_preference = None

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

# App title and description
st.title("🎓 ESB Academic Assistant")
st.markdown("### Multilingual Chat with RAG")

# Sidebar for language selection and settings
with st.sidebar:
    st.header("⚙️ Settings")
    
    if st.session_state.language_preference is None:
        st.info("👈 Please select a language to start chatting")
        language = st.radio(
            "Select your preferred language:",
            ["English", "Français", "عربي"],
            key="lang_select"
        )
        
        if st.button("Continue with " + language):
            st.session_state.language_preference = language.lower() if language != "عربي" else "arabic"
            st.rerun()
    else:
        st.success(f"✅ Current language: {st.session_state.language_preference.capitalize()}")
        
        if st.button("🔄 Change Language"):
            st.session_state.language_preference = None
            st.session_state.messages = [st.session_state.messages[0]]
            st.session_state.chat_history = []
            st.rerun()
        
        if st.button("🗑️ Clear Chat"):
            st.session_state.messages = [st.session_state.messages[0]]
            st.session_state.chat_history = []
            st.rerun()

# Display chat messages
if st.session_state.language_preference is not None:
    # Display chat history
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])
    
    # Chat input
    user_input = st.chat_input("Type your question here...")
    
    if user_input:
        # Check for exit keywords
        exit_keywords = {
            "english": ["bye", "exit", "quit"],
            "french": ["au revoir", "adieu", "quitter"],
            "arabic": ["وداعاً", "خروج"]
        }
        
        lang_key = st.session_state.language_preference
        if lang_key == "français":
            lang_key = "french"
        elif lang_key == "عربي":
            lang_key = "arabic"
        
        if any(keyword in user_input.lower() for keyword in exit_keywords.get(lang_key, [])):
            # Exit messages
            exit_messages = {
                "english": "Goodbye! Thank you for using ESB Academic Assistant. 👋",
                "french": "Au revoir ! Merci d'avoir utilisé l'Assistant Académique ESB. 👋",
                "arabic": "وداعاً! شكراً لاستخدامك مساعد ESB الأكاديمي. 👋"
            }
            
            st.chat_message("user").markdown(user_input)
            st.chat_message("assistant").markdown(exit_messages.get(lang_key, exit_messages["english"]))
            st.session_state.language_preference = None
            st.stop()
        
        # Add user message to session state
        st.session_state.messages.append({"role": "user", "content": user_input})
        st.chat_message("user").markdown(user_input)
        
        # Get AI response
        with st.spinner("🤔 Thinking..."):
            try:
                # Invoke RAG chain
                response = rag_chain(user_input)
                
                # Add assistant response
                st.session_state.messages.append({"role": "assistant", "content": response})
                st.session_state.chat_history.extend([
                    HumanMessage(content=user_input),
                    AIMessage(content=response)
                ])
                
                # Display response
                st.chat_message("assistant").markdown(response)
            except Exception as e:
                st.error(f"❌ Error: {str(e)}")
                st.chat_message("assistant").markdown(f"Sorry, I encountered an error: {str(e)}")


ModuleNotFoundError: No module named 'gradio'

In [ ]:
# 🎓 Streamlit Application Conversion Complete!

## Fichiers créés

1. **`app.py`** - Application Streamlit principale
2. **`run.bat`** - Script de lancement Windows
3. **`run.py`** - Script de lancement Python (multiplateforme)
4. **`requirements.txt`** - Dépendances du projet
5. **`README.md`** - Documentation complète
6. **`GUIDE.md`** - Guide d'utilisation détaillé
7. **`.streamlit/config.toml`** - Configuration Streamlit

## Pour démarrer l'application

### Méthode 1: Script batch (Windows)
```bash
run.bat
```

### Méthode 2: Script Python (Recommandé)
```bash
python run.py
```

### Méthode 3: Direct Streamlit
```bash
streamlit run app.py
```

## Améliorations par rapport à Gradio

✅ **Interface plus moderne** - Streamlit offre une meilleure UI/UX
✅ **Performance meilleure** - Mise en cache native des composants
✅ **Gestion de session** - Historique persistant dans la session
✅ **Sécurité améliorée** - Support natif des secrets
✅ **Scalabilité** - Peut être déployée sur Streamlit Cloud
✅ **Personnalisation** - Configuration facile via config.toml
✅ **Sidebar intégré** - Meilleure organisation des paramètres
✅ **Chat natif** - Interface de chat optimisée

## Caractéristiques Streamlit

- 🌐 Support multilingue (Anglais, Français, Arabe)
- 💬 Interface de chat intuitive
- ⚙️ Barre latérale pour les paramètres
- 🔄 Changement de langue dynamique
- 🗑️ Effacement du chat
- 🎨 Thème personnalisé
- 📱 Interface responsive
- ⚡ Mise en cache pour les performances

## Prochaines étapes

1. Installez les dépendances: `pip install -r requirements.txt`
2. Configurez vos clés API
3. Lancez l'application: `python run.py`
4. Ouvrez http://localhost:8501 dans votre navigateur
5. Sélectionnez votre langue et commencez à discuter!

## Architecture RAG

```
User Query
    ↓
Language Selection & Session Management
    ↓
Retrieval (Chroma Vector Store)
    ↓
Generation (Llama 3.3 via Groq)
    ↓
Response Display & Chat History
```

#gradio a plusieurs utilisateurs



# Évaluation du Chatbot

## Métriques d'évaluation du RAG Chatbot

Ce section contient le code pour évaluer les performances du chatbot avec les métriques d'évaluation courantes.

In [ ]:
%%capture
! pip install -U ragas scikit-learn rouge-score nltk

In [ ]:
# Importer les métriques d'évaluation
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu
from nltk.tokenize import word_tokenize
import nltk

# Télécharger les ressources NLTK nécessaires
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

print("Dépendances d'évaluation installées avec succès!")

In [ ]:
import pandas as pd
import numpy as np
from typing import List, Dict

# Créer une fonction pour calculer ROUGE scores
def calculate_rouge_scores(reference: str, generated: str) -> Dict[str, float]:
    """
    Calcule les scores ROUGE-1, ROUGE-2, et ROUGE-L
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, generated)
    
    return {
        'ROUGE-1': scores['rouge1'].fmeasure,
        'ROUGE-2': scores['rouge2'].fmeasure,
        'ROUGE-L': scores['rougeL'].fmeasure
    }

# Créer une fonction pour calculer BLEU score
def calculate_bleu_score(reference: str, generated: str) -> float:
    """
    Calcule le score BLEU
    """
    reference_tokens = word_tokenize(reference.lower())
    generated_tokens = word_tokenize(generated.lower())
    
    # BLEU utilise une liste de références tokenisées
    return sentence_bleu([reference_tokens], generated_tokens)

# Créer une fonction pour calculer la similarité cosinus
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(reference: str, generated: str) -> float:
    """
    Calcule la similarité cosinus entre deux textes
    """
    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform([reference, generated])
    similarity = cosine_similarity(vectors[0:1], vectors[1:2])[0][0]
    return similarity

# Créer une fonction pour calculer les métriques de longueur
def calculate_length_metrics(reference: str, generated: str) -> Dict[str, float]:
    """
    Calcule les métriques basées sur la longueur du texte
    """
    ref_length = len(reference.split())
    gen_length = len(generated.split())
    
    return {
        'Reference Length': ref_length,
        'Generated Length': gen_length,
        'Length Ratio': gen_length / ref_length if ref_length > 0 else 0
    }

print("Fonctions d'évaluation définies avec succès!")

In [ ]:
# Créer un dataset de test avec des questions et réponses de référence
test_dataset = [
    {
        "question": "What are the study programs provided by ESB?",
        "reference_answer": "ESB offers Bachelor's and Master's degree programs in business administration, finance, and information systems."
    },
    {
        "question": "Quels sont les programmes d'études offerts par l'ESB?",
        "reference_answer": "L'ESB offre des programmes de baccalauréat et de maîtrise en administration des affaires, finance et systèmes d'information."
    },
    {
        "question": "ما هي برامج الدراسة في ESB؟",
        "reference_answer": "تقدم مدرسة إسبريت برامج البكالوريوس والماجستير في إدارة الأعمال والمالية وأنظمة المعلومات."
    }
]

print(f"Dataset de test créé avec {len(test_dataset)} questions")

In [ ]:
# Fonction pour évaluer le chatbot
def evaluate_chatbot(test_dataset: List[Dict]) -> pd.DataFrame:
    """
    Évalue le chatbot en utilisant plusieurs métriques d'évaluation
    
    Args:
        test_dataset: Liste de dictionnaires contenant 'question' et 'reference_answer'
    
    Returns:
        DataFrame contenant les résultats d'évaluation
    """
    results = []
    
    for i, test_case in enumerate(test_dataset):
        question = test_case["question"]
        reference_answer = test_case["reference_answer"]
        
        # Obtenir la réponse du chatbot
        try:
            response = rag_chain.invoke({"input": question, "chat_history": []})
            generated_answer = response["answer"]
        except Exception as e:
            print(f"Erreur lors de l'invocation du RAG chain: {e}")
            generated_answer = ""
        
        # Calculer les métriques
        rouge_scores = calculate_rouge_scores(reference_answer, generated_answer)
        bleu_score = calculate_bleu_score(reference_answer, generated_answer)
        cosine_sim = calculate_cosine_similarity(reference_answer, generated_answer)
        length_metrics = calculate_length_metrics(reference_answer, generated_answer)
        
        # Créer un dictionnaire avec tous les résultats
        result = {
            "Question": question,
            "Reference Answer": reference_answer,
            "Generated Answer": generated_answer,
            "ROUGE-1": rouge_scores['ROUGE-1'],
            "ROUGE-2": rouge_scores['ROUGE-2'],
            "ROUGE-L": rouge_scores['ROUGE-L'],
            "BLEU Score": bleu_score,
            "Cosine Similarity": cosine_sim,
            "Reference Length": length_metrics['Reference Length'],
            "Generated Length": length_metrics['Generated Length'],
            "Length Ratio": length_metrics['Length Ratio']
        }
        
        results.append(result)
        print(f"✓ Question {i+1}/{len(test_dataset)} évaluée")
    
    return pd.DataFrame(results)

print("Fonction d'évaluation du chatbot définie avec succès!")

In [ ]:
# Exécuter l'évaluation du chatbot
evaluation_results = evaluate_chatbot(test_dataset)

# Afficher les résultats sous forme de tableau
print("\n" + "="*100)
print("RÉSULTATS DE L'ÉVALUATION DU CHATBOT")
print("="*100 + "\n")
print(evaluation_results.to_string())

# Calculer les scores moyens
print("\n" + "="*100)
print("SCORES MOYENS")
print("="*100 + "\n")

avg_rouge1 = evaluation_results["ROUGE-1"].mean()
avg_rouge2 = evaluation_results["ROUGE-2"].mean()
avg_rougel = evaluation_results["ROUGE-L"].mean()
avg_bleu = evaluation_results["BLEU Score"].mean()
avg_cosine = evaluation_results["Cosine Similarity"].mean()

print(f"ROUGE-1 Moyen:       {avg_rouge1:.4f}")
print(f"ROUGE-2 Moyen:       {avg_rouge2:.4f}")
print(f"ROUGE-L Moyen:       {avg_rougel:.4f}")
print(f"BLEU Score Moyen:    {avg_bleu:.4f}")
print(f"Cosine Similarity:   {avg_cosine:.4f}")
print("\nNote: Les scores vont de 0 à 1, où 1 est la meilleure note possible.")

In [29]:
# ============================================================================
# 📚 EXPLICATION COMPLÈTE DE L'ARCHITECTURE DU PROJET
# ============================================================================

architecture_doc = """
╔═══════════════════════════════════════════════════════════════════════════╗
║                    ARCHITECTURE COMPLÈTE DU PROJET                        ║
║                      ESB Academic Assistant RAG                           ║
╚═══════════════════════════════════════════════════════════════════════════╝

## 1️⃣ COMPOSANTS PRINCIPAUX

┌─────────────────────────────────────────────────────────────────────┐
│                         DATA & DOCUMENTS                            │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  📄 data_esb.txt (Source)                                          │
│  ├─ Masters disponibles                                            │
│  ├─ Licences/Bachelors disponibles                                 │
│  ├─ Matières par programme                                         │
│  ├─ Emplois après chaque cursus                                    │
│  └─ Secteurs d'activité                                            │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────┐
│                    TEXT PROCESSING PIPELINE                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  1. TextLoader (UTF-8)                                             │
│     └─> Charge le contenu du fichier                              │
│                                                                     │
│  2. RecursiveCharacterTextSplitter                                 │
│     ├─ chunk_size = 800 caractères                                │
│     ├─ chunk_overlap = 100 (continuité entre chunks)              │
│     └─> Génère ~280 chunks                                        │
│                                                                     │
│  3. HuggingFace Embeddings (all-MiniLM-L6-v2)                      │
│     └─> Crée des vecteurs pour chaque chunk                       │
│                                                                     │
│  4. Chroma Vector Store                                            │
│     └─> Stocke les vecteurs pour recherche rapide                 │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘

## 2️⃣ SYSTÈME DE RETRIEVAL (3 niveaux)

┌─────────────────────────────────────────────────────────────────────┐
│                    HYBRID RETRIEVER SYSTEM                           │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  NIVEAU 1️⃣ : PATTERN-BASED SEARCH (Regex)                         │
│  ├─ Priorité: MAXIMUM ⭐⭐⭐                                        │
│  ├─ Utilise: regex.findall() sur le texte brut                    │
│  ├─ Patterns:                                                       │
│  │  ├─ "bachelor programs in esb : [...]"                         │
│  │  ├─ "subjects to study in master X : [...]"                    │
│  │  └─ "jobs you can get choosing X : [...]"                      │
│  └─ Avantage: Résultats EXACTS pour données structurées           │
│                                                                     │
│  NIVEAU 2️⃣ : BM25 KEYWORD SEARCH                                  │
│  ├─ Priorité: MOYENNE ⭐⭐                                         │
│  ├─ Utilise: BM25Retriever (recherche par mots-clés)              │
│  ├─ Excellent pour: Documents structurés                           │
│  └─ Cherche: Mots exacts avec scoring de pertinence               │
│                                                                     │
│  NIVEAU 3️⃣ : SEMANTIC SEARCH (Embeddings)                         │
│  ├─ Priorité: FAIBLE ⭐                                            │
│  ├─ Utilise: Vectorstore Chroma (similarity search)               │
│  ├─ Excellent pour: Questions en langage naturel                  │
│  └─ Comprend: Le sens des mots                                     │
│                                                                     │
│  RÉSULTAT COMBINÉ:                                                 │
│  - Déduplique les résultats                                        │
│  - Retourne top 12 documents les plus pertinents                  │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘

## 3️⃣ GENERATION (LLM)

┌─────────────────────────────────────────────────────────────────────┐
│                         RAG CHAIN                                    │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  Input: User Question                                              │
│    │                                                                │
│    ▼                                                                │
│  Retriever (Hybrid) → Top 12 documents                             │
│    │                                                                │
│    ▼                                                                │
│  Context Assembly → Combine documents into context                 │
│    │                                                                │
│    ▼                                                                │
│  ChatPromptTemplate                                                │
│    ├─ System instructions                                          │
│    ├─ Context: {context}                                           │
│    └─ Question: {question}                                         │
│    │                                                                │
│    ▼                                                                │
│  ChatGroq (llama-3.3-70b-versatile)                                │
│    └─ Génère la réponse basée sur le prompt                       │
│    │                                                                │
│    ▼                                                                │
│  StrOutputParser → Extrait le texte                                │
│    │                                                                │
│    ▼                                                                │
│  Output: Final Response                                            │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘

## 4️⃣ FICHIERS DU PROJET

📦 c:\\Users\\msamet\\Desktop\\Rag\\
├── 📄 data_esb.txt              ⭐ Source de données
├── 📓 chatbot.ipynb             📚 Environnement de développement/test
│                                   ├─ Setup & installations
│                                   ├─ RAG Chain implémentation
│                                   ├─ Tests et debugging
│                                   └─ Métriques d'évaluation
├── 🌐 app.py                    ⭐ PRODUCTION - Interface web
│                                   ├─ Code RAG (même que notebook)
│                                   ├─ Intégration Streamlit
│                                   ├─ Interface utilisateur
│                                   └─ Session management
└── 🌐 app_with_metrics.py       ✨ NOUVEAU - Production + Métriques
                                    ├─ Tout ce qu'app.py fait
                                    ├─ + Calcul ROUGE/BLEU/Cosine
                                    ├─ + Affichage des métriques
                                    └─ + Historique évaluations

## 5️⃣ FLUX DE DONNÉES

DÉVELOPPEMENT (chatbot.ipynb)
└─> Code testé & validé
    └─> Code copié dans app.py
        └─> Exécuté dans Streamlit
            └─> Utilisateur pose question
                └─> RAG Chain retourne réponse

## 6️⃣ COMMENT LE RAG EST UTILISÉ?

chatbot.ipynb:
├─ Cellules 11-15: RAG Chain implémenté directement
│  └─ rag_chain = create_simple_rag_chain(retriever, qa_chain)
│  └─ Utilisé pour tester les questions

app.py:
├─ Fonction load_rag_components() (lignes 28-143)
│  └─ Contient la MÊME logique RAG
│  └─ @st.cache_resource → Exécutée UNE FOIS au startup
│  └─ Retourne la fonction rag_chain
├─ rag_chain = load_rag_components()
│  └─ Assignée à une variable globale
│  └─ Utilisée quand l'utilisateur pose une question:
│     └─ response = rag_chain(user_input)

app_with_metrics.py:
├─ Même structure que app.py
├─ + Fonctions d'évaluation:
│  ├─ calculate_rouge_scores()
│  ├─ calculate_bleu_score()
│  ├─ calculate_cosine_similarity()
│  └─ evaluate_response()
├─ + Session state pour métriques:
│  ├─ evaluation_history
│  └─ reference_answers
└─ + Sidebar pour:
   ├─ Ajouter réponses de référence
   ├─ Afficher historique métriques
   └─ Télécharger rapport CSV

## 7️⃣ RÉSUMÉ DE L'ARCHITECTURE

1. data_esb.txt
   ↓
2. TextLoader + TextSplitter + HuggingFace Embeddings + Chroma
   ↓
3. Hybrid Retriever (Pattern-Based + BM25 + Semantic)
   ↓
4. ChatPromptTemplate + ChatGroq LLM + StrOutputParser
   ↓
5. app.py (Production Interface) / app_with_metrics.py (With Evaluation)
   ↓
6. Streamlit Web Interface
   ↓
7. User Interaction

🎯 POINT CLÉS:
✅ Le RAG est défini UNE FOIS dans load_rag_components()
✅ Il est mis en cache avec @st.cache_resource
✅ Quand l'utilisateur pose une question, on utilise: rag_chain(user_input)
✅ Les métriques sont optionnelles (dans app_with_metrics.py)
"""

print(architecture_doc)


╔═══════════════════════════════════════════════════════════════════════════╗
║                    ARCHITECTURE COMPLÈTE DU PROJET                        ║
║                      ESB Academic Assistant RAG                           ║
╚═══════════════════════════════════════════════════════════════════════════╝

## 1️⃣ COMPOSANTS PRINCIPAUX

┌─────────────────────────────────────────────────────────────────────┐
│                         DATA & DOCUMENTS                            │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  📄 data_esb.txt (Source)                                          │
│  ├─ Masters disponibles                                            │
│  ├─ Licences/Bachelors disponibles                                 │
│  ├─ Matières par programme                                         │
│  ├─ Emplois après chaque cursus                                    │
│  └─ Secteurs